#Phase 1-2

In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install fastf1

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import fastf1
import warnings

warnings.filterwarnings("ignore")
fastf1.Cache.enable_cache('fastf1_cache') 
pd.options.mode.chained_assignment = None

In [5]:
'''#2

# Load a specific session (e.g., 2023 Bahrain Grand Prix - Race)
year, grand_prix, session_type = 2023, 'Bahrain', 'R'
session = fastf1.get_session(year, grand_prix, session_type)
session.load(telemetry=True, laps=True, weather=True)

df = session.laps

# Tag the Race_ID for Group K-Fold Cross Validation later
df['Race_ID'] = f"{year}_{grand_prix}"

# Define the Finishing_Tier dependent variable based on final positions
df['Finishing_Tier'] = np.select(
    [
        df['Position'] <= 3,
        (df['Position'] > 3) & (df['Position'] <= 10)
    ], 
    ['Podium', 'Point Scorer'], 
    default='No Points'
)
'''

'#2\n\n# Load a specific session (e.g., 2023 Bahrain Grand Prix - Race)\nyear, grand_prix, session_type = 2023, \'Bahrain\', \'R\'\nsession = fastf1.get_session(year, grand_prix, session_type)\nsession.load(telemetry=True, laps=True, weather=True)\n\ndf = session.laps\n\n# Tag the Race_ID for Group K-Fold Cross Validation later\ndf[\'Race_ID\'] = f"{year}_{grand_prix}"\n\n# Define the Finishing_Tier dependent variable based on final positions\ndf[\'Finishing_Tier\'] = np.select(\n    [\n        df[\'Position\'] <= 3,\n        (df[\'Position\'] > 3) & (df[\'Position\'] <= 10)\n    ], \n    [\'Podium\', \'Point Scorer\'], \n    default=\'No Points\'\n)\n'

In [8]:
year = 2023
races_to_pull = ['Bahrain', 'Saudi Arabia', 'Australia']
all_clean_laps = []

for grand_prix in races_to_pull:
    print(f"Loading {grand_prix}...")
    session = fastf1.get_session(year, grand_prix, 'R')
    session.load(telemetry=True, laps=True, weather=True)
    
    df_race = session.laps.copy()
    df_race['Race_ID'] = f"{year}_{grand_prix.replace(' ', '')}"
    
    results = session.results[['Abbreviation', 'Status', 'ClassifiedPosition']]
    df_race = df_race.merge(results, left_on='Driver', right_on='Abbreviation', how='left')
    
    # 1. Define Target
    df_race['Finishing_Tier'] = np.select(
        [
            df_race['Position'] <= 3,
            (df_race['Position'] > 3) & (df_race['Position'] <= 10)
        ], 
        ['Podium', 'Point Scorer'], 
        default='No Points'
    )
    
    # 2. DNF Protocol
    dnf_mask = (df_race['Status'] == 'Retired') | (df_race['ClassifiedPosition'] == 'R')
    df_race.loc[dnf_mask, 'Finishing_Tier'] = 'No Points'
    
    # Clean up missing lap times
    df_race = df_race.dropna(subset=['LapTime'])
    df_race.fillna(method='ffill', inplace=True)
    
    all_clean_laps.append(df_race)

# Combine all 3 races into our master 'df'
df = pd.concat(all_clean_laps, ignore_index=True)
print(f"All races stacked successfully! Total rows: {len(df)}")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Loading Bahrain...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Loading Saudi Arabia...


core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_ap

Loading Australia...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '44', '14', '18', '11', '4', '27', '81', '24', '22', '77', '55', '10', '31', '21', '2', '20', '63', '23', '16']


All races stacked successfully! Total rows: 2880


In [9]:
df.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Race_ID,Abbreviation,Status,ClassifiedPosition,Finishing_Tier
0,0 days 01:04:15.902000,VER,1,0 days 00:01:39.019000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:42.414000,...,1.0,False,,False,False,2023_Bahrain,VER,Finished,1,Podium
1,0 days 01:05:53.876000,VER,1,0 days 00:01:37.974000,2.0,1.0,NaT,NaT,0 days 00:00:31.342000,0 days 00:00:42.504000,...,1.0,False,,False,True,2023_Bahrain,VER,Finished,1,Podium
2,0 days 01:07:31.882000,VER,1,0 days 00:01:38.006000,3.0,1.0,NaT,NaT,0 days 00:00:31.388000,0 days 00:00:42.469000,...,1.0,False,,False,True,2023_Bahrain,VER,Finished,1,Podium
3,0 days 01:09:09.858000,VER,1,0 days 00:01:37.976000,4.0,1.0,NaT,NaT,0 days 00:00:31.271000,0 days 00:00:42.642000,...,1.0,False,,False,True,2023_Bahrain,VER,Finished,1,Podium
4,0 days 01:10:47.893000,VER,1,0 days 00:01:38.035000,5.0,1.0,NaT,NaT,0 days 00:00:31.244000,0 days 00:00:42.724000,...,1.0,False,,False,True,2023_Bahrain,VER,Finished,1,Podium


In [10]:
#3 
# Convert tire compounds (Soft, Medium, Hard, Intermediate, Wet) into k-1 dummy variables
df = pd.get_dummies(df, columns=['Compound'], prefix='Tire', drop_first=True)

# Ensure they are integers (0 or 1)
for col in df.columns:
    if col.startswith('Tire_'):
        df[col] = df[col].astype(int)

In [11]:
#4

df['Max_Lap'] = df.groupby('Race_ID')['LapNumber'].transform('max')
df['Fuel_Mass_Proxy'] = 110 - (110 / df['Max_Lap']) * df['LapNumber']
df['Tire_Age'] = df['TyreLife']
mean_tire_age = df['Tire_Age'].mean()
mean_fuel_mass = df['Fuel_Mass_Proxy'].mean()

df['Degradation_x_Fuel'] = (df['Tire_Age'] - mean_tire_age) * (df['Fuel_Mass_Proxy'] - mean_fuel_mass)

In [12]:
# Select final columns for the model
# Dynamically grab tire columns but strictly block 'Tire_Age' duplicates
tire_cols = [col for col in df.columns if col.startswith('Tire_') and col != 'Tire_Age' and not col.endswith('.1')]

features_to_export = [
    'Race_ID', 'Driver', 'LapNumber', 'Finishing_Tier', 
    'Tire_Age', 'Fuel_Mass_Proxy', 'Degradation_x_Fuel'
] + tire_cols

df_clean = df[features_to_export]
df_clean.to_csv('f1_clean_data.csv', index=False)

print(f"Data Prep Complete, Extracted f1_clean_data.csv with {len(df_clean)} rows.")

Data Prep Complete, Extracted f1_clean_data.csv with 2880 rows.
